In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

# Create Spark session
spark = SparkSession.builder.appName("FBData").getOrCreate()

# Define schema for fb_comments_count
comments_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("number_of_comments", IntegerType(), True)
])

# Sample data for fb_comments_count
comments_data = [
    (1, '2023-12-05 10:30:00', 5),
    (2, '2023-12-15 14:20:00', 8),
    (1, '2024-01-10 09:15:00', 3),
    (2, '2023-12-15 14:20:00', 8),
    (3, '2023-12-20 16:45:00', 12),
    (2, '2024-01-05 11:00:00', 6)
]

# Create DataFrame for fb_comments_count
df_comments = spark.createDataFrame(comments_data, schema=comments_schema)

# Define schema for fb_active_users
users_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("country", StringType(), True)
])

# Sample data for fb_active_users
users_data = [
    (1, 'USA'),
    (2, 'UK'),
    (3, 'Canada')
]

# Create DataFrame for fb_active_users
df_users = spark.createDataFrame(users_data, schema=users_schema)

In [ ]:
# Import your libraries
import pyspark
from pyspark.sql.functions import col,dense_rank,lead,month,year
from pyspark.sql.window import Window

In [ ]:
# Start writing code
fb_comments_count_df=fb_comments_count.withColumn("month",month(col('created_at'))).withColumn("year",year(col("created_at")))
fb_users_comments_count_df=fb_comments_count_df.alias("fc").join(fb_active_users.alias("fb"),col("fc.user_id")==col("fb.user_id"),"inner").select("number_of_comments","month","year","country").groupBy("month","year","country")\
    .agg({"number_of_comments":"sum"}).select("month","year","country",col("sum(number_of_comments)").alias("total_comments")).filter("month=1 or month=12")
window_lead=Window.partitionBy("country").orderBy(col("year").asc(),col("month").asc())
window_rn=Window.partitionBy("month","year").orderBy(col("total_comments").desc())
fb_country_rank=fb_users_comments_count_df.withColumn("rn",dense_rank().over(window_rn))
fb_country_progress_df=fb_country_rank.withColumn("next_rn",lead(col("rn")).over(window_lead)).filter("next_rn<rn and next_rn is not null").select("country")


In [ ]:
fb_country_progress_df.show()